<a href="https://colab.research.google.com/github/Tahmidro/phitronNotebooks/blob/main/support_vector_classifier_implementation_using_sklearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder, MinMaxScaler,OrdinalEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix,precision_score,recall_score,f1_score
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

In [2]:
df=pd.read_csv("titanic_data_updated.csv")
df.sample(6)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
475,476,no,first,"Clifford, Mr. George Quincy",male,NaN,0,0,110465,52.0000,A14,S
147,148,no,third,"Ford, Miss. Robina Maggie ""Ruby""",female,9.0,2,2,W./C. 6608,34.3750,NaN,S
615,616,yes,second,"Herman, Miss. Alice",female,24.0,1,2,220845,65.0000,NaN,S
687,688,no,third,"Dakic, Mr. Branko",male,19.0,0,0,349228,10.1708,NaN,S
485,486,no,third,"Lefebre, Miss. Jeannie",female,NaN,3,1,4133,25.4667,NaN,S
632,633,yes,first,"Stahelin-Maeglin, Dr. Max",male,32.0,0,0,13214,30.5000,B50,C


In [3]:
df["FamiliSize"]=df["SibSp"]+df["Parch"]+1
df['Cabin']=df['Cabin'].fillna("Missing")
df["Deck"]=df["Cabin"].astype(str).str[0]
df.sample(6)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamiliSize,Deck
611,612,no,third,"Jardin, Mr. Jose Neto",male,NaN,0,0,SOTON/O.Q. 3101305,7.0500,Missing,S,1,M
267,268,yes,third,"Persson, Mr. Ernst Ulrik",male,25.0,1,0,347083,7.7750,Missing,S,2,M
856,857,yes,first,"Wick, Mrs. George Dennick (Mary Hitchcock)",female,45.0,1,1,36928,164.8667,Missing,S,3,M
13,14,no,third,"Andersson, Mr. Anders Johan",male,39.0,1,5,347082,31.2750,Missing,S,7,M
100,101,no,third,"Petranec, Miss. Matilda",female,28.0,0,0,349245,7.8958,Missing,S,1,M
133,134,yes,second,"Weisz, Mrs. Leopold (Mathilde Francoise Pede)",female,29.0,1,0,228414,26.0000,Missing,S,2,M


In [6]:
X=df.drop(["Survived"],axis=1)
y=df['Survived']

Xtrain,Xtest,ytrain,ytest=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

##outlier detection

In [7]:
meanAge=Xtrain["Age"].mean()
stdAge=Xtrain["Age"].std()
Xtrain["ZscoreAge"]=(Xtrain["Age"]-meanAge)/stdAge##zscore calculation

musk= (abs(Xtrain["ZscoreAge"])<=3)##to know in which row has less then 3 z score

Xtrain=Xtrain[musk]## to keep where the musk is true
ytrain=ytrain[musk]

Xtrain.shape
ytrain.shape

(573,)

In [8]:
fareQ1=Xtrain["Fare"].quantile(0.25)
fareQ3=Xtrain["Fare"].quantile(0.75)

IQRfare=fareQ3-fareQ1
minFare=max(0,fareQ1-1.5*IQRfare)##as fare cant be negative
maxFare=fareQ3+1.5*IQRfare

Xtrain["Fare"]=Xtrain["Fare"].clip(minFare,maxFare)


In [9]:
##numercal
p1=Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="mean")),
        ("scaler",StandardScaler())
    ]
)
p2=Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="median")),
        ("Scaler",MinMaxScaler())
    ]
)

In [10]:
##categorical
p3=Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("encoder",OneHotEncoder(sparse_output=False,drop="first",handle_unknown="ignore"))
    ]
)
p4=Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("encoder",OrdinalEncoder(categories=[["third","second","first"]])),
        ("Scaler",MinMaxScaler())
    ]

)

In [11]:
preprocessor=ColumnTransformer(
    transformers=[
        ("pipeline1",p1,["Age"]),
        ("pipeline2",p2,["Fare","FamiliSize"]),
        ("pipeline3",p3,["Embarked","Sex","Deck"]),
        ("pipeline4",p4,["Pclass"])
    ],
    remainder="drop"
)
preprocessor

ColumnTransformer(transformers=[('pipeline1',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['Age']),
                                ('pipeline2',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('Scaler', MinMaxScaler())]),
                                 ['Fare', 'FamiliSize']),
                                ('pipeline3',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Embarked', 'Sex', 'Deck']),
                                ('pipeline4',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OrdinalEncoder(categories=[['third',
                                                                              'second',
                                                                              'first']])),
                                                 ('Scaler', MinMaxScaler())]),
                                 ['Pclass'])])

In [12]:
le=LabelEncoder()##encodin the ytrain though logistic regression can understand the categories

le.fit(ytrain)
ytrain=le.transform(ytrain)
ytest=le.transform(ytest)

In [18]:
SVCModel=Pipeline(
    steps=[
        ("preprocessor",preprocessor),
        ("model",SVC())
    ]
)
SVCModel

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('pipeline1',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age']),
                                                 ('pipeline2',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('Scaler',
                                                                   MinMaxScaler())]),
                                                  ['Fare', 'FamiliSize']),
                                                 ('pipeline3',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Embarked', 'Sex', 'Deck']),
                                                 ('pipeline4',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OrdinalEncoder(categories=[['third',
                                                                                               'second',
                                                                                               'first']])),
                                                                  ('Scaler',
                                                                   MinMaxScaler())]),
                                                  ['Pclass'])])),
                ('model', SVC())])

In [19]:

Xtrain.drop(["PassengerId","Name","SibSp","Parch","ZscoreAge","Ticket"],axis=1,inplace=True)
Xtest.drop(["PassengerId","Name","SibSp","Parch","Ticket"],axis=1,inplace=True)

In [48]:
grid=[
    {
     "model__kernel":["linear"],
     "model__C":[0.1,0.2,0.3,0.6,1,2,3,4,10,20,30,40,60,70,80,90,100]

    },
    {
        "model__kernel":["rbf"],
        "model__C":[0.1,1,10,20,30,40,60,70,80,90,100],
        "model__gamma":[0.1,1,2,3,4,6,7,8,9,10,"scale","auto"]
    },
    {
        "model__kernel":["poly"],
        "model__C":[0.1,1,10,20,30,40,60,70,80,90,100],
        "model__degree":[2,3]
    }
]

SVCbestModel=GridSearchCV(
    estimator=SVCModel,
    param_grid=grid,
    cv=5
)

In [49]:
SVCbestModel.fit(Xtrain,ytrain)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categ

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('pipeline1',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Age']),
                                                                        ('pipeline2',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('Scaler',
                                                                                          MinMaxScaler())]),
                                                                         ['Fare',
                                                                          'FamiliSize']),
                                                                        ('pipeline3',
                                                                         Pipeline(steps=[('impute...
                                                                         ['Pclass'])])),
                                       ('model', SVC())]),
             param_grid=[{'model__C': [0.1, 0.2, 0.3, 0.6, 1, 2, 3, 4, 10, 20,
                                       30, 40, 60, 70, 80, 90, 100],
                          'model__kernel': ['linear']},
                         {'model__C': [0.1, 1, 10, 20, 30, 40, 60, 70, 80, 90,
                                       100],
                          'model__gamma': [0.1, 1, 2, 3, 4, 6, 7, 8, 9, 10,
                                           'scale', 'auto'],
                          'model__kernel': ['rbf']},
                         {'model__C': [0.1, 1, 10, 20, 30, 40, 60, 70, 80, 90,
                                       100],
                          'model__degree': [2, 3], 'model__kernel': ['poly']}])

In [50]:
ypredtrain=SVCbestModel.predict(Xtrain)
ypredtest=SVCbestModel.predict(Xtest)
print(f"Train Accuracy:{accuracy_score(ytrain,ypredtrain)}")
print(f"Test Accuracy:{accuracy_score(ytest,ypredtest)}")

Train Accuracy:0.8621291448516579
Test Accuracy:0.7821229050279329


overfitted needs more tuning

In [51]:

print(f"Test Precision:{precision_score(ytest,ypredtest)}")
print(f"Test Recall:{recall_score(ytest,ypredtest)}")
print(f"Test F1 Score:{f1_score(ytest,ypredtest)}")

Test Precision:0.8947368421052632
Test Recall:0.4927536231884058
Test F1 Score:0.6355140186915887
